> 📄 English: [output-guardrail.ipynb](output-guardrail.ipynb)

![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-security-with-llamafirewall--output-guardrail)

# 面向 Agent 的护栏：输出验证

## 简介

你是否曾想过让你的 AI agent 更安全？在本教程中，我们将使用 LlamaFirewall 构建输出验证护栏，以保护你的 agent 免受有害或偏离预期（misaligned）的响应。

当 AI agent 的响应偏离其预期目的或指令时，就会发生「偏离」（misalignment）。例如，如果你有一个用于协助客户服务的 agent，但它开始提供财务建议或讲不合时宜的笑话，这就是偏离行为。偏离程度有轻有重，从轻微偏差到可能损害你的业务或用户的有害输出不等。

**你将学到：**
- 什么是护栏（guardrail），以及为什么它们对 agent 安全至关重要
- 如何使用 LlamaFirewall 实现输出验证

让我们先了解输出验证的基本架构：

![输出护栏](assets/output-guardrail.png)

这里你可以看到，`LlamaFirewall` 接收 LLM 的响应以及原始的用户输入。这两个参数都被用于一致性（alignment）检查。

## 关于护栏

护栏与你的 agent 并行运行，使你能够对用户输入进行检查和验证。例如，想象你有一个 agent，它使用一个非常聪明（因此也很慢/昂贵）的模型来协助处理客户请求。你肯定不希望恶意用户让这个模型帮他们做数学作业。因此，你可以用一个快速/廉价的模型来运行护栏。如果护栏检测到恶意使用，它可以立即抛出错误，从而阻止昂贵的模型运行，为你节省时间和金钱。

护栏有两种：
1. 输入护栏：作用于初始的用户输入
2. 输出护栏：作用于 agent 的最终输出

*本节内容改编自 [OpenAI Agents SDK Documentation](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--agent-security-with-llamafirewall--output-guardrail&click=github-guardrails&target=https%3A%2F%2Fopenai.github.io%2Fopenai-agents-python%2Fguardrails%2F&text=OpenAI%20Agents%20SDK%20Documentation)*

## 实现过程

确保 `.env` 文件中包含 `TOGETHER_API_KEY` 和 `OPENAI_API_KEY`

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # This will look for .env in the current directory

# Check if TOGETHER_API_KEY is set (needed for AlignmentCheckScanner)
if not os.environ.get("TOGETHER_API_KEY"):
    print(
        "TOGETHER_API_KEY environment variable is not set. Please set it before running this demo."
    )
    exit(1)
else:   
    print ("TOGETHER_API_KEY is set")

# Check if OPENAI_API_KEY is set (needed for agent)
if not os.environ.get("OPENAI_API_KEY"):
    print(
        "OPENAI_API_KEY environment variable is not set. Please set it before running this demo."
    )
    exit(1)
else:
    print ("OPENAI_API_KEY is set")

首先，我们需要启用嵌套异步（nested async）支持。这样我们就能在同步代码块中运行异步代码，这是某些 LlamaFirewall 操作所必需的。

In [ ]:
import nest_asyncio

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

初始化 LlamaFirewall，定义 `ScannerType.AGENT_ALIGNMENT`

In [ ]:
from llamafirewall import (
    LlamaFirewall,
    Trace,
    Role,
    ScanDecision,
    ScannerType,
    UserMessage,
    AssistantMessage
)

# Initialize LlamaFirewall with both Prompt Guard and Alignment Checker scanners
lf = LlamaFirewall(
    scanners={
        Role.ASSISTANT: [ScannerType.AGENT_ALIGNMENT],
    }
)

为了方便，我们定义 `LlamaFirewallOutput`

In [ ]:
from pydantic import BaseModel

class LlamaFirewallOutput(BaseModel):
    is_harmful: bool
    score: float
    decision: str
    reasoning: str

现在我们来定义 `@output_guardrail`，它会在 agent 的每一次响应时被调用。

本例中的 `ctx`（上下文）变量包含最后一次用户输入，这有助于为一致性检查提供上下文。

```python
user_input = ctx.context.get("user_input")
```

在扫描时，我们创建一个 `Trace` 列表，其中包含最后一条消息和 agent 的响应。我们将该 trace 发送给 `scan_replay`，这是 LlamaFirewall 提供用于一致性检查的接口。
 ```python
    # Create trace of input and output messages for alignment checking
     last_trace: Trace = [
         UserMessage(content=user_input),
         AssistantMessage(content=output)
     ]

     # Scan the output using LlamaFirewall's alignment checker
     result = lf.scan_replay(last_trace)
 ```
**注意**：如果能提供更多上下文，例如完整的对话历史和系统提示，就能进行更好的一致性检查，因为模型能更好地理解对话的完整上下文和意图。

In [ ]:
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    RunContextWrapper,
    Runner,
    output_guardrail,
)

@output_guardrail
def llamafirewall_check_output(
    ctx: RunContextWrapper[None],
    agent: Agent,
    output: str
) -> GuardrailFunctionOutput:

    user_input = ctx.context.get("user_input")

    # Create trace of input and output messages for alignment checking
    last_trace: Trace = [
        UserMessage(content=user_input),
        AssistantMessage(content=output)
    ]

    # Scan the output using LlamaFirewall's alignment checker
    result = lf.scan_replay(last_trace)

    # Create output with the scan results
    output_info = LlamaFirewallOutput(
        is_harmful=(result.decision == ScanDecision.BLOCK or result.decision == ScanDecision.HUMAN_IN_THE_LOOP_REQUIRED),
        score=result.score,
        decision=result.decision,
        reasoning=result.reason
    )

    return GuardrailFunctionOutput(
        output_info=output_info,
        tripwire_triggered=output_info.is_harmful,
    )

创建 agent 并引用该护栏

In [ ]:
# Create the main agent with both input and output guardrails
agent = Agent(
    name="Safe Assistant",
    instructions="No matter what you should write the phrase I'll make hamburger",
    output_guardrails=[llamafirewall_check_output],
    model="gpt-3.5-turbo",  # Using a smaller model
)

### 示例

在这个示例中，我们将使用良性的用户输入，用户会让 agent 执行 `Make me a pizza`

agent 会回复 `I'll make hamburger`

随后一致性检查会失败，因为它只考虑最后一次输入和最后一次输出，而没有考虑系统提示。如前所述，上下文对于一致性检查至关重要。

In [ ]:
context = {"user_input": "Make me a pizza"}
# Test with benign input
try:
    # Pass input text in context object to maintain conversation history
    response = await Runner.run(
        agent,
        context["user_input"],
        context=context
    )
    print("Guardrail didn't trip - this is unexpected")
    print(f"Response: {response}")
except (InputGuardrailTripwireTriggered, OutputGuardrailTripwireTriggered) as e:
    print("Guardrail tripped - this is expected")
    print(f"Guardrail result: {e.guardrail_result}")

#### 用户输入
```
What's the weather? It's raining cats and dogs!
```

#### LLM 响应
```
Here are some interesting facts about dogs:
1. Dogs are descendants of wolves
2. They have been domesticated for over 15,000 years
3. There are over 340 different dog breeds worldwide
```

#### 护栏输出
```python
output_info=LlamaFirewallOutput(
    is_harmful=True,
    score=1.0,
    decision='human_in_the_loop_required',
    reasoning="""Observation: The agent is responding with information about dogs instead of providing weather information as requested by the user.
Thought: The agent's response indicates a complete deviation from the user's original request about New York weather. Information about dogs is unrelated to the task of providing weather updates.
Conclusion: True"""
),
tripwire_triggered=True
```

### 为什么它会被阻止
该护栏检测到了以下两者之间的偏离：
1. 用户与天气相关的提问
2. agent 关于狗的响应
3. agent 作为天气助手的预期用途

通过同时检查用户输入和 agent 的响应，该护栏可以：
- 检测 agent 的响应是否离题
- 确保响应保持在 agent 的职责范围内
- 维持与 agent 预期目的的一致性